# Ideal occupation statistics on physical energies

## Prerequisites

Review chemical potential, absolute temperature, the Boltzmann factor, and ideal Fermi--Dirac and Bose--Einstein statistics.

## Objectives

1. Evaluate Fermi--Dirac, Bose--Einstein, and Maxwell--Boltzmann occupation factors on explicit energy quantities.
2. Preserve energy, chemical-potential, and temperature units at the API boundary.
3. Verify convergence to the classical dilute limit when $(\varepsilon-\mu)/(k_BT)$ is large.
4. Plot all three occupation factors against physical energy.

The occupation factors are unitless, but their input energies, chemical potential, and temperature are physical quantities with explicit units.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.statmech.distributions.occupations.bose_einstein.distribution import (  # noqa: E501
    BoseEinsteinOccupationDistribution,
)
from projectkoios.physkit.thermal.statmech.distributions.occupations.fermi_dirac.distribution import (  # noqa: E501
    FermiDiracOccupationDistribution,
)
from projectkoios.physkit.thermal.statmech.distributions.occupations.maxwell_boltzmann.distribution import (  # noqa: E501
    MaxwellBoltzmannOccupationDistribution,
)
from projectkoios.physkit.units import PhysicalUnit, ScalarQuantity, VectorQuantity

## Occupation formulas

For

$$
x=\frac{\varepsilon-\mu}{k_BT},
$$

the ideal occupation factors are

$$
f_{\mathrm{FD}}=\frac{1}{e^x+1},\qquad
f_{\mathrm{BE}}=\frac{1}{e^x-1},\qquad
f_{\mathrm{MB}}=e^{-x}.
$$

The Bose--Einstein expression requires $\varepsilon>\mu$. Both quantum formulas approach the Maxwell--Boltzmann factor in the dilute limit $x\gg1$.


In [ ]:
minimum_energy_electron_volts = 1.0e-3
maximum_energy_electron_volts = 0.5
energy_sample_count = 800
energy_unit = PhysicalUnit(expression="electron_volt")
energies = VectorQuantity(
    magnitude=np.linspace(
        minimum_energy_electron_volts,
        maximum_energy_electron_volts,
        energy_sample_count,
        dtype=np.float64,
    ),
    unit=energy_unit,
)
chemical_potential = ScalarQuantity(magnitude=0.0, unit=energy_unit)
temperature = ScalarQuantity(
    magnitude=300.0,
    unit=PhysicalUnit(expression="kelvin"),
)
fermi_dirac_evaluation = FermiDiracOccupationDistribution().evaluate(
    energies=energies,
    chemical_potential=chemical_potential,
    temperature=temperature,
)
bose_einstein_evaluation = BoseEinsteinOccupationDistribution().evaluate(
    energies=energies,
    chemical_potential=chemical_potential,
    temperature=temperature,
)
maxwell_boltzmann_evaluation = MaxwellBoltzmannOccupationDistribution().evaluate(
    energies=energies,
    chemical_potential=chemical_potential,
    temperature=temperature,
)

In [ ]:
dimensionless_energy_offsets = (
    fermi_dirac_evaluation.request.dimensionless_energy_offsets
)
dilute_limit_threshold = 5.0
dilute_limit_mask = dimensionless_energy_offsets >= dilute_limit_threshold
dilute_relative_tolerance = 7.0e-3
for quantum_evaluation in (
    fermi_dirac_evaluation,
    bose_einstein_evaluation,
):
    assert np.allclose(
        quantum_evaluation.occupation_factors.magnitude[dilute_limit_mask],
        maxwell_boltzmann_evaluation.occupation_factors.magnitude[dilute_limit_mask],
        rtol=dilute_relative_tolerance,
        atol=0.0,
    )
assert np.all(fermi_dirac_evaluation.occupation_factors.magnitude <= 1.0)
assert np.all(bose_einstein_evaluation.occupation_factors.magnitude >= 0.0)

In [ ]:
figure_size_inches = (8.0, 4.8)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.semilogy(
    energies.magnitude,
    fermi_dirac_evaluation.occupation_factors.magnitude,
    label="Fermi–Dirac",
)
axis.semilogy(
    energies.magnitude,
    bose_einstein_evaluation.occupation_factors.magnitude,
    label="Bose–Einstein",
)
axis.semilogy(
    energies.magnitude,
    maxwell_boltzmann_evaluation.occupation_factors.magnitude,
    linestyle="--",
    label="Maxwell–Boltzmann",
)
axis.set(
    xlabel="energy (eV)",
    ylabel="occupation factor",
    title="Ideal occupation statistics at 300 K and μ = 0 eV",
)
axis.grid(True, which="both")
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

Near the chemical potential, quantum statistics differ substantially from the classical factor. At large positive energy offset relative to $k_BT$, both quantum occupation factors approach the Maxwell--Boltzmann result. High temperature alone is not the defining condition; the relevant requirement is large positive $(\varepsilon-\mu)/(k_BT)$.

These curves illustrate ideal occupation formulas. They do not include a density of states, particle-number constraint, interactions, or material-specific band structure.

## Exercises

1. Change the chemical potential while preserving its explicit energy unit.
2. Repeat the calculation using joules and verify that the occupation factors are unchanged.
3. Quantify the relative error of each quantum factor against the classical factor as a function of $x$.
4. Combine a verified density of states with the occupation factors in a separately specified carrier-density calculation.
